# Module 1 — Risk Profiling

**Key concept:** We separate *risk capacity* (objective, financial ability to absorb losses) from *risk tolerance* (psychological willingness). Most retail advisors conflate them — capacity should act as the hard ceiling.

Output: a risk persona (Conservative / Balanced / Aggressive) + a numeric target return that feeds the optimizer in Module 3.

In [ ]:
# ── Setup (Colab) ────────────────────────────────────────────────
!pip install -q numpy pandas PyPortfolioOpt plotly yfinance fpdf2
import os
if not os.path.exists('robo-advisor'):
    !git clone https://github.com/parinmody30/robo-advisor.git
else:
    !git -C robo-advisor pull

import sys
sys.path.insert(0, 'robo-advisor/src')
DATA_DIR = 'robo-advisor/data'
print('Setup complete.')

In [ ]:
from risk_profiler import build_profile, RiskProfile

## 1A. Capacity inputs (objective financial facts)

In [ ]:
# ── Edit these values ──────────────────────────────────────────
age                    = 28
income_stability       = 3   # 1=unstable, 2=moderate, 3=stable
dependents             = 0   # number of financial dependents
emergency_fund_months  = 6   # months of expenses covered
investment_horizon_yrs = 15  # years until you need the money
# ──────────────────────────────────────────────────────────────

## 1B. Tolerance inputs (behavioural questions)

In [ ]:
tolerance_answers = {
    # If your portfolio dropped 20% in a month, you would:
    # 1=sell everything  2=hold  3=buy more
    'market_drop_reaction': 3,

    # Your investing experience:
    # 1=none  2=some mutual funds  3=stocks/ETFs/bonds
    'past_investing_exp': 2,

    # If your portfolio was down 15%, your sleep would be:
    # 1=severely affected  2=somewhat uneasy  3=fine, I trust the process
    'loss_sleep': 3,

    # You are comfortable with portfolio swings of:
    # 1=<5%  2=5-15%  3=>15%
    'volatility_comfort': 2,

    # Your goal timeline is:
    # 1=rigid  2=somewhat flexible  3=flexible
    'goal_flexibility': 2,
}

## 1C. Generate risk profile

In [ ]:
profile = build_profile(
    age=age,
    income_stability=income_stability,
    dependents=dependents,
    emergency_fund_months=emergency_fund_months,
    investment_horizon_yrs=investment_horizon_yrs,
    tolerance_answers=tolerance_answers,
)

print('=' * 40)
print(f'  Risk Persona     : {profile.persona}')
print(f'  Combined Score   : {profile.combined_score} / 10')
print(f'  Capacity Score   : {profile.capacity_score} / 10')
print(f'  Tolerance Score  : {profile.tolerance_score} / 10')
print(f'  Target Return    : {profile.target_return*100:.0f}% p.a.')
print('=' * 40)
if profile.tolerance_score > profile.capacity_score:
    print('\n⚠️  Tolerance capped to capacity — you WANT more risk than you can AFFORD.')
else:
    print('\n✓  Capacity and tolerance are aligned.')

## 1D. Visualise score breakdown

In [ ]:
import plotly.graph_objects as go, json, os

fig = go.Figure(go.Bar(
    x=['Capacity', 'Tolerance', 'Combined'],
    y=[profile.capacity_score, profile.tolerance_score, profile.combined_score],
    marker_color=['#4CAF50', '#2196F3', '#FF9800'],
    text=[f'{v:.1f}' for v in [profile.capacity_score, profile.tolerance_score, profile.combined_score]],
    textposition='outside',
))
fig.update_layout(
    title=f'Risk Profile: {profile.persona} (Target {profile.target_return*100:.0f}% p.a.)',
    yaxis=dict(range=[0, 11], title='Score'),
    template='plotly_white',
)
fig.show()

os.makedirs(DATA_DIR, exist_ok=True)
with open(f'{DATA_DIR}/risk_profile.json', 'w') as f:
    json.dump(profile.__dict__, f, indent=2)
print(f'Profile saved to {DATA_DIR}/risk_profile.json')